## STRESS

In [ ]:
import os
from pathlib import Path

# Get the current notebook's directory
notebook_dir = Path().absolute()

# Get the parent directory of the current notebook's folder
parent_dir = notebook_dir.parent

# Change the working directory to the parent folder
os.chdir(parent_dir)

# Print to verify the change
print("New Working Directory:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np

# Carica i file
ps_df = pd.read_csv('DATASET_DIST/ps_definitivi.csv', encoding='ISO-8859-1', sep=',', quotechar='"')
access_df = pd.read_csv('DATASET_DIST/total_accessibility_clean.csv')

# Colonne popolazione
pop_columns = ['Y_LT15', 'Y1564', 'Y_GE65']

# Converti tutto in stringa
def to_int_str(x):
    try:
        return str(int(float(x)))
    except:
        return ''

# Conversione standardizzata
ps_df['Codice_struttura'] = ps_df['Codice_struttura'].apply(to_int_str)
ps_df['Subcodice'] = ps_df['Subcodice'].apply(to_int_str)

access_df['ps_code_1'] = access_df['ps_code_1'].apply(to_int_str)
access_df['ps_code_2'] = access_df['ps_code_2'].apply(to_int_str)
access_df['ps_ped_code'] = access_df['ps_ped_code'].apply(to_int_str)

access_df['ps_subcode_1'] = access_df['ps_subcode_1'].apply(to_int_str)
access_df['ps_subcode_2'] = access_df['ps_subcode_2'].apply(to_int_str)
access_df['ps_ped_subcode'] = access_df['ps_ped_subcode'].apply(to_int_str)

# Inizializza colonne
for col in pop_columns:
    ps_df[col] = 0

# Funzione per confrontare subcodici in modo flessibile
def subcode_match(a, b):
    valid_nan = ['', '0', 'NA']
    return (a == b) or (a in valid_nan and b in valid_nan)

# Ciclo su ciascun pronto soccorso
for idx, row in ps_df.iterrows():
    codice = row['Codice_struttura']
    subcodice = row['Subcodice']
    is_pediatric = row['PS.PEDIATRICO'] == 'SI'  # Controllo se ospedale è pediatrico

    if is_pediatric:
        # Ricerca solo su ospedali pediatrici
        mask_ped = (access_df['ps_ped_code'] == codice) & access_df['ps_ped_subcode'].apply(lambda x: subcode_match(x, subcodice))
        matching_ped = access_df[mask_ped]

        # Somma solo la popolazione sotto i 15 anni
        if 'Y_LT15' in matching_ped.columns:
            ps_df.at[idx, 'Y_LT15'] += matching_ped['Y_LT15'].sum()
    else:
        # Ricerca su ospedali di base (ps_code_1 e ps_code_2)
        mask1 = (access_df['ps_code_1'] == codice) & access_df['ps_subcode_1'].apply(lambda x: subcode_match(x, subcodice))
        mask2 = (access_df['ps_code_2'] == codice) & access_df['ps_subcode_2'].apply(lambda x: subcode_match(x, subcodice))
        matching_regular = access_df[mask1 | mask2]

        # Somma tutte le fasce di popolazione
        for col in pop_columns:
            if col in matching_regular.columns:
                ps_df.at[idx, col] += matching_regular[col].sum()

# Esporta
ps_df.to_csv('DATASET_DIST/ps_definitivi_con_popolazioni_fasce.csv', index=False)


In [ ]:
import pandas as pd
import numpy as np

# Carica il dataset
df = pd.read_csv('DATASET_DIST/ps_definitivi_con_popolazioni_fasce.csv', encoding = 'ISO-8859-1', sep = ',')

# Dataset popolazione
df_pop = pd.read_csv('DATASET_DIST/popolazione_redistribuita.csv')


# Lista delle colonne da trattare
cols_to_fill = ['N..BIANCHI','N..VERDI','N..GIALLI','N..ROSSI',
    'X..BIANCHI.RIC', 'X..VERDI.RIC', 'X..GIALLI.RIC', 'X..ROSSI.RIC',
    'ACCESSI.PS.TOT', 'DECESSI',
    'avg_TO_Acuti',
    'PL.DS.Effettivi', 'PL.DS.Previsti', 'PL.Totali.Previsti',
    'PL.DH.Effettivi', 'PL.DH.Previsti'
]

# Converto a numerico e gestisco NA
for col in cols_to_fill:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # assicura numerico
    df[col] = df.groupby('LIVELLO.EMERGENZA')[col].transform(lambda x: x.fillna(x.median()))  # mediana per ospedale
    df[col] = df[col].fillna(df[col].median())  # se ancora NaN, usa mediana globale

# Prevenzione divisione per zero
df['ACCESSI.PS.TOT'] = df['ACCESSI.PS.TOT'].replace(0, np.nan)



In [ ]:
print(df.columns.tolist())


In [ ]:
# Calcolo percentuali
df['percent_bianchi'] = np.where(df['ACCESSI.PS.TOT'] != 0, df['N..BIANCHI'] / df['ACCESSI.PS.TOT'], np.nan)
df['percent_verdi'] = np.where(df['ACCESSI.PS.TOT'] != 0, df['N..VERDI'] / df['ACCESSI.PS.TOT'], np.nan)
df['percent_gialli'] = np.where(df['ACCESSI.PS.TOT'] != 0, df['N..GIALLI'] / df['ACCESSI.PS.TOT'], np.nan)
df['percent_rossi'] = np.where(df['ACCESSI.PS.TOT'] != 0, df['N..ROSSI'] / df['ACCESSI.PS.TOT'], np.nan)

# Funzione MinMax scaler
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

# Calcolo e normalizzazione componenti
C1 = (1/5) * minmax(df['percent_bianchi'] * df['X..BIANCHI.RIC'] * df['avg_TO_Acuti'])
C2 = (1/5) * minmax(df['percent_verdi'] * df['X..VERDI.RIC'] * df['avg_TO_Acuti'])
C3 = (1/5) * minmax(df['percent_gialli'] * df['X..GIALLI.RIC'] * df['avg_TO_Acuti'])
C4 = (1/5) * minmax(df['percent_rossi'] * df['X..ROSSI.RIC'] * df['avg_TO_Acuti'])
C5 = (1/10) * minmax(df['DECESSI'] / df['ACCESSI.PS.TOT'])

# Calcolo C6 e C7 con gestione NaN
df['C6'] =(df['PL.DS.Effettivi'] / df['PL.DS.Previsti'])
df['C7'] =(df['PL.DH.Effettivi'] / df['PL.DH.Previsti'])

df['C6'] = df.groupby('LIVELLO.EMERGENZA')['C6'].transform(lambda x: x.fillna(x.median()))
df['C7'] = df.groupby('LIVELLO.EMERGENZA')['C7'].transform(lambda x: x.fillna(x.median()))

# Normalizzazione anche di C6 e C7
C6 =  (1/20) * minmax(df['C6'])
C7 =  (1/20) * minmax(df['C7'])

# Calcolo indice stress
df['stress_raw'] = C1 + C2 + C3 + C4 + C5 + C6 + C7

df['stress'] = minmax((0.33 * df['Y_LT15'] + 0.27 * df['Y1564'] + 0.40 * df['Y_GE65']) / (df['PL.Totali.Previsti']) * df['stress_raw'])

# Rimuovo C6 e C7 prima di salvare
df.drop(columns=['C6', 'C7', 'stress_raw','percent_bianchi','percent_rossi', 'percent_gialli','percent_verdi'], inplace=True)

# Verifica righe con molte colonne NaN
print("Righe con più di 10 NaN:", df[df.isnull().sum(axis=1) > 3].shape[0])

# Salva il risultato
df.to_csv('DATASET_DIST/ps_stress.csv', index=False)

# Visualizza i primi risultati
print(df[['DESCRIZIONE', 'stress']].head(600))

In [ ]:
cols_in_formula = [
    'X..BIANCHI.RIC', 'X..VERDI.RIC', 'X..GIALLI.RIC', 'X..ROSSI.RIC',
    'ACCESSI.PS.TOT', 'avg_TO_Acuti', 'DECESSI',
    'PL.DS.Effettivi', 'PL.DS.Previsti',
    'PL.DH.Effettivi', 'PL.DH.Previsti'
]

print("NaN per ogni colonna coinvolta nel calcolo dell'indice stress:")
print(df[cols_in_formula].isna().sum())


In [ ]:
import pandas as pd

# Carica il CSV
df = pd.read_csv("total_accessibility_clean.csv")

# Funzione di calcolo del rischio
def compute_risk(row):
    t1 = row['time_travel_ps1']
    t2 = row['time_travel_ps2']
    t3 = row['time_travel_ped']
    s1 = row['stress_ps_1']
    s2 = row['stress_ps_2']
    s3 = row['stress_ped']

    total_time = t1 + t2 + t3
    if total_time == 0:
        return (s1 + s2 + s3) / 3  # fallback se tutti i tempi sono zero

    # Pesi basati su tempi di percorrenza
    p1 = t1 / total_time
    p2 = t2 / total_time
    p3 = t3 / total_time

    return p1 * s1 + p2 * s2 + p3 * s3

# Applica la funzione
df['risk_index'] = df.apply(compute_risk, axis=1)

# Esporta il risultato
df.to_csv("total_accessibility_with_risk.csv", index=False)

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Carica il dataset
df = pd.read_csv("total_accessibility_with_risk.csv")

# Colonne dei tempi di percorrenza
time_cols = ['time_travel_airport', 'time_travel_port', 'time_travel_station']

# Normalizzazione dei tempi (più tempo = più rischio)
scaler = MinMaxScaler()
df_norm = pd.DataFrame(scaler.fit_transform(df[time_cols]), columns=[f'norm_{col}' for col in time_cols])

# Aggiungi le colonne normalizzate al dataframe originale
df = pd.concat([df, df_norm], axis=1)

# Definizione dei pesi
weights = {
    'norm_time_travel_airport': 0.4,
    'norm_time_travel_port': 0.2,
    'norm_time_travel_station': 0.4
}

# Calcolo dell'access_risk_index
df['access_risk_index'] = sum(df[col] * weight for col, weight in weights.items()) / sum(weights.values())

# Calcolo dell'indice finale
alpha = 0.75
beta = 0.25
df['final_index'] = alpha * df['risk_index'] + beta * df['access_risk_index']

# Esportazione del risultato finale
df.to_csv("total_accessibility_with_final_index.csv", index=False)
